# Matrix Alignment



## Purpose

The purpose of this notebook is to determine whether every participant listed in the cohort metadata corresponds to exactly one FCM file.

The audit checks participant identifiers and FCM filenames without inspecting the contents of the matrices.

The required correspondence is:

**225 metadata participants ↔ 225 unique FCM files**

The audit will identify missing, additional, duplicated, malformed, or incorrectly named FCM files.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# Load Data 

In [2]:
# 1. Loading Data
fcm_data = Path(
    "../data/external/Curvature-FCN-Aging/DATA/FCM"
)

cohort_metadata = pd.read_csv(
    "../data/external/Curvature-FCN-Aging/DATA/cohort_information.tsv",
    sep="\t",
    dtype={"sub_id": "string"},
)
# 2. Gather directory data including extension
entries = []
naming_convention = "fcm_sub_" 
for entry in sorted(fcm_data.iterdir()):
    is_file = entry.is_file()
    is_system_file = is_file and entry.name.startswith(".")
    has_prefix = entry.name.startswith(naming_convention)

    extension = entry.suffix.lower() if is_file else "N/A"

    entries.append(
        {
            "Name": entry.name,
            "Type": "File" if is_file else "Folder",
            "Extension": extension if extension else "No Extension",
            "Prefix": has_prefix,
            "Is_System": is_system_file,
        }
    )

df = pd.DataFrame(entries)

## FCM directory inventory

The inventory checks the ***total number*** of `directory entries`, `regular files`, `subdirectories`, `file extensions`, `unexpected file types`, and `common system files`. This ensures that unexpected entries are not silently excluded from the participant-to-matrix comparison.

Only directory names and filenames are inspected at this stage. 

In [3]:
def directory_check(df):
    report = {}

    regular_files = df["Type"].eq("File")
    system_files = df["Is_System"]
    non_system_file = regular_files & ~system_files
    txt_files = df["Extension"].eq(".txt") 

    # 1. Total number of directory entries and Regular Files
    report["Total Number of entries"] = len(df)
    report["Number of Regular Files"] = int(regular_files.sum())

    # 2. Number of .txt Files
    report["Number of non-system .txt files"] = int((non_system_file & txt_files).sum())
    

    # 3. Number of files following FCM naming convention
    expected_prefix = (
        non_system_file & txt_files & df['Prefix']
    )
    report["Files with expected prefix and .txt extension"] = int(expected_prefix.sum())

    # 4. Any unexpected file extensions (files that are not .txt and not system files)
    unexpected_ext_df = (
        non_system_file & ~txt_files
    )

    unexpected_extensions = df.loc[
        unexpected_ext_df, "Name"
    ].tolist()

    report["Unexpected file extensions"] = (unexpected_extensions if unexpected_extensions else "None")
    
    # 5. Any unexpected subdirectories
    unexpected_directories = df.loc[
        df["Type"].eq("Folder"), "Name"
    ].tolist()

    report["Unexpected subdirectories"] = (
        unexpected_directories if unexpected_directories else "None"
    )
    # 6. Any system files such as .DS_Store
    unexpected_system_file = df.loc[system_files, "Name"].tolist()

    report["System files found"] = (
        unexpected_system_file if unexpected_system_file else "None"
    )

    
    return pd.Series(report)


directory_check(df)

Total Number of entries                           225
Number of Regular Files                           225
Number of non-system .txt files                   225
Files with expected prefix and .txt extension     225
Unexpected file extensions                       None
Unexpected subdirectories                        None
System files found                               None
dtype: object

### Interpretation

The FCM directory inventory identified `225 regular files` and no unexpected subdirectories, file extensions, or system files. All 225 files use the expected `.txt` extension.

## FCM filename validation

The FCM directory was inspected at the filename level to determine whether the available matrices follow the expected naming convention.

The expected filename format is:

`fcm_sub_<subject-ID>.txt`

For example:

`fcm_sub_32301.txt`

The validation checks whether each filename:

* starts with the required `fcm_sub_` prefix;
* ends with the `.txt` extension;
* contains a participant ID between the prefix and extension;
* contains only numeric characters in the participant ID;
* can be parsed unambiguously into the original participant ID.




In [4]:
def pattern_validation(df):

    report = {}

    candidate_file = (
        df["Type"].eq("File")
        & df["Extension"].eq(".txt")
        & ~df["Is_System"]
    )

    extracted_ids = df["Name"].str.extract(
        r"^fcm_sub_([0-9]+)\.txt$",
        expand=False,
    ).astype("string")

    valid_filename = candidate_file & extracted_ids.notna()

    df['Participant_ID'] = extracted_ids.where(valid_filename, pd.NA)

    df['is_valid_FCM_Filename'] = valid_filename

    malformed_filename = candidate_file & ~valid_filename

    report['Non-system .txt files checked'] = int(candidate_file.sum())
    report["Valid Filenames (Perfect Match)"] = int(valid_filename.sum())
    report["Malformed / Unparseable Filenames"] = int(malformed_filename.sum())


    return pd.Series(report)

pattern_validation(df)

Non-system .txt files checked        225
Valid Filenames (Perfect Match)      225
Malformed / Unparseable Filenames      0
dtype: int64

### Interpretation

The FCM filenames were checked against the expected `fcm_sub_<subject-ID>.txt`. Valid filenames were parsed to recover the original participant IDs, while any malformed filenames were recorded separately for investigation.

##  Data Integrity and Normalization

In [5]:
def normalization(df):
    report = {}

    df['Participant_ID'] = df['Participant_ID'].astype("string").str.strip()

    valid_file = df['is_valid_FCM_Filename']

    df['Normalized ID'] = pd.Series(
        pd.NA,
        index = df.index,
        dtype = 'string'
    )
    

    df.loc[valid_file, 'Normalized ID'] = (
        'sub-'
        + df.loc[
            valid_file, 
            "Participant_ID",
        ].str.zfill(6)
    )

    valid_format = df['Normalized ID'].str.fullmatch(r"sub-[0-9]{6}",
        na=False)

    report["Matrix IDs normalized"] = int(valid_file.sum())
    report["Canonical IDs with valid format"] = int(valid_format.sum())
    report["Canonical IDs with invalid format"] = int((valid_file & ~valid_format).sum())

    return  pd.Series(report)

normalization(df)

Matrix IDs normalized                225
Canonical IDs with valid format      225
Canonical IDs with invalid format      0
dtype: int64

## interpretation
Participant identifiers are treated as `strings` 

The metadata participant IDs are loaded as `strings`, and participant IDs parsed from FCM filenames are also retained as `strings`. No participants are renumbered, and gaps in the original identifier sequence are preserved.

The original participant identifier supplied by Yadav remains unchanged. A canonical identifier is created separately by zero-padding the original ID to six digits and adding the sub- prefix.

The canonical conversion is checked for collisions to ensure that distinct participant IDs do not resolve to the same canonical identifier.


# Check uniqueness of matrix IDs.

In [6]:
def check_uniqueness(df):
    report = {}

    valid_matrix_df = df.loc[df["is_valid_FCM_Filename"]].copy()

    parsed_ids = valid_matrix_df['Participant_ID'].dropna()

    report["Number of valid FCM files"] = len(valid_matrix_df)
    report["Number of parsed matrix IDs"] = len(parsed_ids)
    report["Number of unique parsed matrix IDs"] = (parsed_ids.nunique())
    report['Duplicated IDs in filenames'] = int(parsed_ids.duplicated().sum())

    numeric_ids = parsed_ids.map(int)
    report['Canonical Collision Count'] = int(parsed_ids.nunique() - numeric_ids.nunique())

    normalized_ids = valid_matrix_df['Normalized ID'].dropna()
    report["Duplicated normalized IDs"] = int(normalized_ids.duplicated().sum())

    return pd.Series(report)

check_uniqueness(df)

Number of valid FCM files             225
Number of parsed matrix IDs           225
Number of unique parsed matrix IDs    225
Duplicated IDs in filenames             0
Canonical Collision Count               0
Duplicated normalized IDs               0
dtype: int64

## interpretation

The participant IDs extracted from the validated FCM filenames were checked for uniqueness and one-to-one representation.

The audit compares the total number of FCM files with the number of parsed and unique participant IDs. It also identifies participant IDs appearing in more than one filename, exact duplicate filenames, and any collisions introduced by canonical ID conversion.


## Cross-Referencing and Set Validation

In [7]:
def cross_reference_directories(df, cohort_metadata):
    report = {}

    valid_matrix_df = df.loc[df['is_valid_FCM_Filename']]
    valid_fcm_ids = valid_matrix_df['Normalized ID'].dropna().unique()

    clean_meta_ids = cohort_metadata['sub_id'].astype("string").str.strip()
    normalized_meta_ids = 'sub-' + clean_meta_ids.str.zfill(6)
    valid_meta_ids = normalized_meta_ids.dropna().unique()

    fcm_series = pd.Series(valid_fcm_ids)
    meta_series = pd.Series(valid_meta_ids)

    # Direction 1: Metadata without matrices
    metadata_only = ~meta_series.isin(fcm_series)
    report["Metadata without matrices"] = int(metadata_only.sum())

    # Direction 2: Matrices without metadata
    matrix_only = ~fcm_series.isin(meta_series)
    report["Matrices without metadata"] = int(matrix_only.sum())

    # Intersection: Matched
    matched_participants = fcm_series.isin(meta_series)
    report["Matched participants"] = int(matched_participants.sum())

    return pd.Series(report)

cross_reference_directories(df, cohort_metadata)

    

Metadata without matrices      0
Matrices without metadata      0
Matched participants         225
dtype: int64

## interpretation

The participant IDs from the cohort metadata were compared with the participant IDs parsed from the FCM filenames.

The comparison was performed in both directions to identify participants present in the metadata without a corresponding FCM file and FCM files whose participant IDs are absent from the metadata.


# Confirm one-to-one cardinality

In [8]:
def mapping_validation(df, cohort_metadata):
    report = {}
    
    valid_files_df = df.loc[df["is_valid_FCM_Filename"], ["Name", "Normalized ID"]].copy()
    cohort_metadata['Normalized ID'] = 'sub-' + cohort_metadata['sub_id'].astype("string").str.strip().str.zfill(6)
    valid_meta_df = cohort_metadata[["sub_id", "Normalized ID"]].copy()
    
   
    valid_files_df['file_instances'] = valid_files_df.groupby('Normalized ID')['Normalized ID'].transform('count')
    valid_meta_df['meta_instances'] = valid_meta_df.groupby('Normalized ID')['Normalized ID'].transform('count')
    
    
    mapping_df = pd.merge(
        valid_meta_df, 
        valid_files_df, 
        on="Normalized ID", 
        how="outer"
    )
    
    is_one_to_one = (mapping_df['meta_instances'] == 1) & (mapping_df['file_instances'] == 1)
    
    
    mapping_df['Mapping_Status'] = "Unmatched / Discrepancy"
    mapping_df.loc[is_one_to_one, 'Mapping_Status'] = "Valid 1:1 Match"
    
  
    report["Metadata row count equals 1 for all"] = "Yes" if (valid_meta_df['meta_instances'] == 1).all() else "No"
    report["Matrix-file count equals 1 for all"] = "Yes" if (valid_files_df['file_instances'] == 1).all() else "No"
    

    report["Perfect 1:1 Matched Participants"] = int(is_one_to_one.sum())
    
   
    has_cardinality_violations = (mapping_df['meta_instances'] > 1) | (mapping_df['file_instances'] > 1)
    report["Cardinality Violations Exist"] = "Yes" if has_cardinality_violations.any() else "No"
    
    return pd.Series(report), mapping_df


mapping_report, final_mapping_table = mapping_validation(df, cohort_metadata)
print(mapping_report)


Metadata row count equals 1 for all    Yes
Matrix-file count equals 1 for all     Yes
Perfect 1:1 Matched Participants       225
Cardinality Violations Exist            No
dtype: object


## interpretation

The participant-level mapping was checked to confirm that each participant occurs exactly once in the cohort metadata and exactly once among the FCM files.

The audit verifies that every metadata participant has one corresponding matrix file, that every matrix participant has one metadata record, and that the participant identifiers represented in both sources agree.

The final mapping status is recorded as `one_to_one` when all three conditions are satisfied.


## The participant-level alignment

In [12]:
def create_alignment_table(df, cohort_metadata):
    meta_df = cohort_metadata.copy()
    meta_df['Original subject ID'] = meta_df['sub_id'].astype(str).str.strip()
    meta_df['Canonical participant ID'] = 'sub-' + meta_df['Original subject ID'].str.zfill(6)
    
   
    candidate_file = (
    df["Type"].eq("File")
    & df["Extension"].eq(".txt")
    & ~df["Is_System"]
    )

    file_df = df.loc[candidate_file].copy()
   
    file_df['Original subject ID'] = file_df['Participant_ID'].fillna(file_df['Name']).astype("string").str.strip()
    
    
    file_df['Canonical participant ID'] = file_df['Normalized ID'].fillna(
        file_df['Original subject ID'].apply(lambda x: f"sub-{x.zfill(6)}" if x.isalnum() else f"invalid-{x}")
    )

   
    all_canonical_ids = sorted(set(meta_df['Canonical participant ID']).union(set(file_df['Canonical participant ID'])))
    
    alignment_rows = []
    
   
    for canon_id in all_canonical_ids:
       
        meta_subset = meta_df[meta_df['Canonical participant ID'] == canon_id]
        metadata_present = not meta_subset.empty
        
       
        file_subset = file_df[file_df['Canonical participant ID'] == canon_id]
        matrix_present = not file_subset.empty
        
       
        matrix_file_count = len(file_subset)
        matrix_filename = ", ".join(file_subset['Name'].tolist()) if matrix_present else np.nan
        
       
        if metadata_present:
            orig_id = meta_subset['Original subject ID'].iloc[0]
        else:
            orig_id = file_subset['Original subject ID'].iloc[0]
            
        
        if matrix_present:
            filename_valid = file_subset['is_valid_FCM_Filename'].all()
        else:
            filename_valid = False
            

        if matrix_present and not filename_valid:
            status = "invalid_filename"
        elif matrix_file_count > 1:
            status = "duplicate_matrix"
        elif metadata_present and not matrix_present:
            status = "metadata_only"
        elif matrix_present and not metadata_present:
            status = "matrix_only"
        elif metadata_present and matrix_present and matrix_file_count == 1 and filename_valid:
            status = "matched"
        else:
            status = "anomaly"

        
        alignment_rows.append({
            "Original subject ID": orig_id,
            "Canonical participant ID": canon_id,
            "Metadata present": metadata_present,
            "Matrix present": matrix_present,
            "Matrix-file count": matrix_file_count,
            "Matrix filename": matrix_filename,
            "Filename valid": filename_valid,
            "Alignment status": status
        })
        
 
    alignment_table = pd.DataFrame(alignment_rows)
    return alignment_table

# the alignment report table
alignment_report = create_alignment_table(df, cohort_metadata)

print("=== AGGREGATE ALIGNMENT RESULTS ===")
print(alignment_report['Alignment status'].value_counts())

output_dir = Path("../data/interim")
output_dir.mkdir(parents=True, exist_ok=True)
alignment_report.to_csv(output_dir / "participant_alignment_table.csv", index=False)



=== AGGREGATE ALIGNMENT RESULTS ===
Alignment status
matched    225
Name: count, dtype: int64


## interpretation

A participant-level alignment table was created to record the correspondence between the cohort metadata and the FCM filenames.

The table contains one row for each participant ID found in either source. It records the original participant ID, the canonical participant ID, whether the participant is present in the metadata, whether an FCM file is present, the number of FCM files associated with the participant, the corresponding filename, and the final alignment status.

The complete table is retained locally because it contains participant identifiers. It is not intended for inclusion in the repository.


## Discrepancies

In [10]:
discrepancies = alignment_report[alignment_report['Alignment status'] != 'matched']

print("\n=== DISCREPANCY AUDIT REPORT ===")
if discrepancies.empty:
    print("Success: Zero discrepancies found. All 225 participants are perfectly matched!")
else:
    print(f"Warning: Found {len(discrepancies)} file alignment issue(s):")
    print(discrepancies[['Canonical participant ID', 'Matrix filename', 'Alignment status']])



=== DISCREPANCY AUDIT REPORT ===
Success: Zero discrepancies found. All 225 participants are perfectly matched!


### Interpretation

The discrepancy audit identified no records with an alignment status other than `matched`. This indicates that every participant represented in the alignment table has a corresponding FCM file and that no metadata-only, matrix-only, duplicate-matrix, or invalid-filename cases were detected.

The participant-level metadata-to-FCM filename alignment therefore passed the discrepancy check.
